In [ ]:
import torch
from lightning.pytorch.callbacks import Callback

class VisualLoggingCallback(Callback):
    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        # Only log for the first batch of the validation epoch
        if batch_idx == 0:
            img_input, img_target = batch
            
            # Switch to eval mode for prediction
            pl_module.eval()
            with torch.no_grad():
                img_pred = pl_module(img_input.to(pl_module.device))
            pl_module.train()

            # Create a grid: Input, Prediction, and Target side-by-side
            # (Assumes images are normalized; you might need to un-normalize)
            comparison = torch.cat([img_input[:4], img_pred[:4], img_target[:4]], dim=3)
            
            # Log to your preferred logger (e.g., TensorBoard or WandB)
            trainer.logger.experiment.add_images('Visual_Progress', comparison, trainer.global_step)

# Add it to your Trainer
# trainer = L.Trainer(callbacks=[VisualLoggingCallback()])

In [ ]:
import torch
import torch.nn.functional as F
import lightning as L
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

class Image2ImageModel(L.LightningModule):
    def __init__(self, model, lr=1e-4):
        super().__init__()
        self.model = model  # e.g., a timm model or custom UNet
        self.lr = lr
        
        # Metrics for I2I are specialized
        self.psnr = PeakSignalNoiseRatio()
        self.ssim = StructuralSimilarityIndexMeasure()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        # 1. Prepare Data (Input -> Target)
        img_input, img_target = batch 
        
        # 2. Forward Pass
        img_pred = self(img_input)
        
        # 3. Calculate Loss (MSE or L1 are standard for I2I)
        loss = F.mse_loss(img_pred, img_target)
        
        # 4. Logging
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img_input, img_target = batch
        img_pred = self(img_input)
        
        val_loss = F.mse_loss(img_pred, img_target)
        
        # Update and Log Metrics
        self.psnr(img_pred, img_target)
        self.ssim(img_pred, img_target)
        
        self.log("val_loss", val_loss, prog_bar=True)
        self.log("val_psnr", self.psnr, on_epoch=True)
        self.log("val_ssim", self.ssim, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)
    
